# Chapter 10, Exercise 2: Simulating wait-k and computing Average Lagging

> **How to use this notebook.** Open it in Google Colab (File > Upload notebook, or the Colab badge on the companion website), then run the cells from top to bottom. Runtime > Change runtime type lets you pick a GPU when one is recommended below. Everything else runs on the free CPU tier.

## The exercise

**Chapter 10, Exercise 2.** Explain the quality-latency trade-off in simultaneous translation and the wait-k policy, then write a Python script that simulates wait-k with k = 3: given an Arabic transcript arriving one word per second, emit the English tokens with the fixed lag and compute the Average Lagging over the utterance. Explain what your Average Lagging value reflects.

## Requirements

No GPU is required; the notebook runs on Colab's free CPU runtime.


<a href="https://colab.research.google.com/github/arabic-speech-book/arabic-speech-book.github.io/blob/main/docs/solutions/Chapter_10_Exercise_02.ipynb" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

## 1. The policy in one paragraph

A simultaneous system must decide, at every moment, whether to **read** one more source word or **write** one more target word. Waiting gives context and better translations; writing early cuts delay but risks committing before the meaning is clear (Section 10.3). **Wait-k** is the simplest fixed policy: read k source words, then alternate write, read, write, read, so that the output stays k words behind the input. The lag is fixed in advance rather than learned from the content.

## 2. Simulation: Arabic source arriving one word per second, k = 3

The source is the kiosk request of the opening scene with a time expression added. The "translation" is a word-aligned English gloss (we are simulating the *schedule*, not a real MT model), so each English token is emitted as soon as the policy allows.

In [1]:
import pandas as pd

source = ["احجز", "رحلة", "إلى", "الرياض", "غدا", "صباحا", "من", "فضلك"]      # 8 Arabic words, 1 word per second
# a monotone English rendering with one token per source word (illustrative; a real MT model reorders)
target = ["book", "a-flight", "to", "Riyadh", "tomorrow", "morning", "please", "."]

def wait_k_schedule(source, target, k):
    """Return, for each target token j (1-based), g(j) = number of source words read before writing it."""
    S, T = len(source), len(target)
    g = []
    for j in range(1, T + 1):
        g.append(min(j + k - 1, S))     # wait-k: token j is written after reading j+k-1 source words (capped at S)
    return g

k = 3
g = wait_k_schedule(source, target, k)
rows = []
for j, (tok, gj) in enumerate(zip(target, g), start=1):
    rows.append({"target token j": j, "token": tok, "source words read g(j)": gj,
                 "emitted at time (s)": gj, "source available at that time": " ".join(source[:gj])})
pd.DataFrame(rows)

,target token j,token,source words read g(j),emitted at time (s),source available at that time
0,1,book,3,3,احجز رحلة إلى
1,2,a-flight,4,4,احجز رحلة إلى الرياض
2,3,to,5,5,احجز رحلة إلى الرياض غدا
3,4,Riyadh,6,6,احجز رحلة إلى الرياض غدا صباحا
4,5,tomorrow,7,7,احجز رحلة إلى الرياض غدا صباحا من
5,6,morning,8,8,احجز رحلة إلى الرياض غدا صباحا من فضلك
6,7,please,8,8,احجز رحلة إلى الرياض غدا صباحا من فضلك
7,8,.,8,8,احجز رحلة إلى الرياض غدا صباحا من فضلك


Reading the table: nothing is written for the first 3 seconds; then one English token appears every second, three source words behind. With one source word per second, "emitted at time" equals g(j) seconds. In reality the source words are grouped into audio chunks and the times are read off the speech, but the schedule is the same.

## 3. Average Lagging (AL)

Average Lagging (Ma et al., 2019) compares the system's read/write schedule with an ideal simultaneous policy that stays exactly in step with the source. With S source words, T target tokens and g(j) the number of source words read before writing target token j:

$$AL = \frac{1}{\tau} \sum_{j=1}^{\tau} \left( g(j) - \frac{j-1}{T/S} \right), \qquad \tau = \min\{j : g(j) = S\}$$

The sum runs only until the first target token that was written after the *whole* source had been read (τ), so that tokens emitted after the speaker finished do not inflate the lag. The unit is source words (here also seconds).

In [2]:
def average_lagging(g, S, T):
    tau = next(j for j, gj in enumerate(g, start=1) if gj == S)
    r = T / S                                       # target/source length ratio
    lags = [g[j-1] - (j-1)/r for j in range(1, tau + 1)]
    return sum(lags)/tau, tau, lags

AL, tau, lags = average_lagging(g, len(source), len(target))
print(f"S = {len(source)} source words, T = {len(target)} target tokens, k = {k}")
print(f"tau (first token written after the full source was read) = {tau}")
print("per-token lags:", [round(x, 2) for x in lags])
print(f"Average Lagging = {AL:.2f} source words  (= {AL:.2f} seconds at one word per second)")

for kk in [1, 2, 3, 5, 8]:
    gk = wait_k_schedule(source, target, kk)
    print(f"k={kk}: AL = {average_lagging(gk, len(source), len(target))[0]:.2f}")

S = 8 source words, T = 8 target tokens, k = 3
tau (first token written after the full source was read) = 6
per-token lags: [3.0, 3.0, 3.0, 3.0, 3.0, 3.0]
Average Lagging = 3.00 source words  (= 3.00 seconds at one word per second)
k=1: AL = 1.00
k=2: AL = 2.00
k=3: AL = 3.00
k=5: AL = 5.00
k=8: AL = 8.00


## 4. What the value reflects

With equal source and target lengths (r = 1), the lag of token j under wait-k is g(j) minus (j minus 1) = k for every token until the source runs out, so **AL equals k = 3 source words, that is 3 seconds at this speaking rate**. The value says: on average, each English token was produced when the system had read three more Arabic words than an idealized in-step policy would have needed. It is a property of the *schedule*, not of translation quality: a k = 3 system that translates badly and one that translates well have the same AL, which is why AL must always be reported together with BLEU, chrF or COMET (Table 10.4). Three details change the number and must be stated: whether latency includes computation time (this simulation is non-computation-aware), how the audio was segmented into "words" or chunks (here an idealized one word per second), and the length ratio r, which for real Arabic-to-English output is above 1 (English uses more tokens than Arabic, Table 4.4), so the ideal policy is credited with writing faster and the measured AL drops below k. The last line of the cell shows the trade-off directly: k = 1 minimizes AL at the cost of translating each word with almost no right context, and k = 8 is offline translation with AL equal to the whole utterance.